In [17]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import Dataset, DataLoader


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [19]:
DATA_DIR  = "DATA"
IMAGE_DIR = os.path.join(DATA_DIR, "images")
AUDIO_DIR = os.path.join(DATA_DIR, "audio")
VIDEO_DIR = os.path.join(DATA_DIR, "raw_videos")


In [20]:
emotion_map = {
    "01": 0, "02": 1, "03": 2, "04": 3,
    "05": 4, "06": 5, "07": 6, "08": 7
}


In [21]:
records = []

for root, _, files in os.walk(VIDEO_DIR):
    for f in files:
        if not f.lower().endswith(".mp4"):
            continue

        parts = f.replace(".mp4", "").split("-")
        if len(parts) != 7 or parts[0] != "01":
            continue

        wav = os.path.join(AUDIO_DIR, f.replace(".mp4", ".wav"))
        if not os.path.exists(wav):
            continue

        records.append({
            "audio": wav,
            "emotion": emotion_map[parts[2]],
            "actor": int(parts[-1])
        })

df = pd.DataFrame(records)
len(df)


1440

In [22]:
train_df = df[df["actor"] <= 20].reset_index(drop=True)
val_df   = df[df["actor"] > 20].reset_index(drop=True)

len(train_df), len(val_df)


(1200, 240)

In [23]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=1024,
    hop_length=512,
    n_mels=64
)

db_transform = torchaudio.transforms.AmplitudeToDB()


In [24]:
MAX_LEN = 128  # fixed time dimension

class AudioEmotionDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        waveform, sr = torchaudio.load(row["audio"])
        if sr != 16000:
            waveform = torchaudio.functional.resample(waveform, sr, 16000)

        mel = mel_transform(waveform)          # [1, 64, T]
        mel = db_transform(mel)

        # ---- FIX LENGTH ----
        T = mel.shape[-1]

        if T < MAX_LEN:
            pad = MAX_LEN - T
            mel = torch.nn.functional.pad(mel, (0, pad))
        else:
            mel = mel[:, :, :MAX_LEN]

        # normalize
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)

        label = torch.tensor(row["emotion"], dtype=torch.long)
        return mel, label


In [25]:
tmp_ds = AudioEmotionDataset(train_df)
mel, label = tmp_ds[0]

mel.shape, label


(torch.Size([1, 64, 128]), tensor(0))

In [26]:
train_audio_ds = AudioEmotionDataset(train_df)
val_audio_ds   = AudioEmotionDataset(val_df)

train_audio_loader = DataLoader(
    train_audio_ds,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_audio_loader = DataLoader(
    val_audio_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


In [27]:
class AudioCNN_Temporal(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),      # ↓ freq, ↓ time

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),      # ↓ freq, ↓ time

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )

        self.classifier = nn.Linear(64, 8)

    def forward(self, x):
        """
        x: [B, 1, 64, 128]
        """
        x = self.features(x)       # [B, 64, F, T]

        # --- SAFE temporal pooling ---
        x = x.mean(dim=2)          # avg over frequency → [B, 64, T]
        x = x.mean(dim=2)          # avg over time      → [B, 64]

        return self.classifier(x)


In [28]:
audio_model = AudioCNN_Temporal().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(audio_model.parameters(), lr=1e-3)


In [29]:
def train_audio_epoch(model, loader):
    model.train()
    correct = total = 0
    loss_sum = 0

    for mel, labels in loader:
        mel = mel.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(mel)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return loss_sum / len(loader), correct / total


In [30]:
@torch.no_grad()
def validate_audio(model, loader):
    model.eval()
    correct = total = 0

    for mel, labels in loader:
        mel = mel.to(device)
        labels = labels.to(device)

        outputs = model(mel)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    return correct / total


In [31]:
a, _ = train_audio_ds[0]
b, _ = train_audio_ds[1]
a.shape, b.shape


(torch.Size([1, 64, 128]), torch.Size([1, 64, 128]))

In [33]:
audio_model

AudioCNN_Temporal(
  (features): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
  )
  (classifier): Linear(in_features=64, out_features=8, bias=True)
)

In [16]:
EPOCHS = 20

for epoch in range(EPOCHS):
    train_loss, train_acc = train_audio_epoch(audio_model, train_audio_loader)
    val_acc = validate_audio(audio_model, val_audio_loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )


Epoch 1/20 | Loss: 2.0101 | Train Acc: 0.245 | Val Acc: 0.271
Epoch 2/20 | Loss: 1.9050 | Train Acc: 0.343 | Val Acc: 0.208
Epoch 3/20 | Loss: 1.8146 | Train Acc: 0.388 | Val Acc: 0.287
Epoch 4/20 | Loss: 1.7409 | Train Acc: 0.411 | Val Acc: 0.300
Epoch 5/20 | Loss: 1.6627 | Train Acc: 0.419 | Val Acc: 0.312
Epoch 6/20 | Loss: 1.6134 | Train Acc: 0.435 | Val Acc: 0.308
Epoch 7/20 | Loss: 1.5860 | Train Acc: 0.441 | Val Acc: 0.263
Epoch 8/20 | Loss: 1.5184 | Train Acc: 0.476 | Val Acc: 0.392
Epoch 9/20 | Loss: 1.4809 | Train Acc: 0.480 | Val Acc: 0.321
Epoch 10/20 | Loss: 1.4560 | Train Acc: 0.461 | Val Acc: 0.408
Epoch 11/20 | Loss: 1.4361 | Train Acc: 0.490 | Val Acc: 0.329
Epoch 12/20 | Loss: 1.4069 | Train Acc: 0.510 | Val Acc: 0.275
Epoch 13/20 | Loss: 1.3899 | Train Acc: 0.513 | Val Acc: 0.317
Epoch 14/20 | Loss: 1.3557 | Train Acc: 0.502 | Val Acc: 0.308
Epoch 15/20 | Loss: 1.3243 | Train Acc: 0.542 | Val Acc: 0.388
Epoch 16/20 | Loss: 1.3040 | Train Acc: 0.530 | Val Acc: 0.279
E